# Goldman SDD Algorithm — Python Port
### Google Earth Engine Python API

Direct Python translation of the Goldman Snow Disappearance Day (SDD) script.
Produces one 500-m GeoTIFF per year (2024, 2025) exported to Google Drive.

Variable names, logic, and order match the original JavaScript exactly.

## Cell 1 — Authenticate

In [1]:
# !pip install earthengine-api --quiet   # uncomment if needed

import ee
import geopandas as gpd

ee.Authenticate()
ee.Initialize(project='ee-jandrewgoldman')   # ← replace

print('Earth Engine initialised.')

Earth Engine initialised.


## Cell 2 — User Inputs

Edit this cell. All variable names match the Goldman script.

In [11]:
# ── Region asset ──────────────────────────────────────────────────────────────
# Your study region as a GEE FeatureCollection asset.
# Goldman uses individual fire perimeters — we use the full region boundary
# and export one raster covering the entire area.
#Read local file and reproject to WGS84
gdf = gpd.read_file("../data/processed_study_zones/study_zones_wgs.geojson").to_crs("EPSG:4326")
region_fc = ee.FeatureCollection(gdf.__geo_interface__)
region_geom = region_fc.geometry()

# ── Years to run ──────────────────────────────────────────────────────────────
YEARS = [2024, 2025]

# ── Goldman user inputs (named identically to original) ──────────────────────
# Goldman: var wyStartDay = '-10-01'
# We use a spring melt window instead of a full water year to reduce compute.
# Adjust these dates for your region's typical melt season.
wyStartDay    = '-02-01'   # start of primary search window (Goldman: '-10-01')
wyEndDay      = '-08-01'   # end of primary search window

# Goldman: var NDSI_threshold = '15'
NDSI_threshold = 15

# Goldman: var low_value = 0.07
low_value = 0.07

# Goldman: var prior_days = 5
prior_days = 5

# Goldman: var minNoSnowdays = 5
minNoSnowdays = 5

# Goldman: var projection = 'EPSG:4326'
projection = 'EPSG:4326'

# ── Export folder ─────────────────────────────────────────────────────────────
imagesFolder = 'SDDperYear'   # Goldman: var imagesFolder = 'SDDperFire'

print('User inputs set.')
print(f'  NDSI_threshold : {NDSI_threshold}')
print(f'  low_value      : {low_value}')
print(f'  prior_days     : {prior_days}')
print(f'  minNoSnowdays  : {minNoSnowdays}')
print(f'  Window         : {wyStartDay} → {wyEndDay} + 30 adv. days')

User inputs set.
  NDSI_threshold : 15
  low_value      : 0.07
  prior_days     : 5
  minNoSnowdays  : 5
  Window         : -02-01 → -08-01 + 30 adv. days


## Cell 3 — Load Region Asset

Goldman loops over individual fire perimeters and clips the SDD image per fire.
We load the full region boundary here and export one regional raster instead.

In [12]:
region_geom = region_fc.geometry()

print('Region loaded.')

Region loaded.


## Cell 4 — Define `getMetrics(wyr)`

Direct Python port of Goldman's `getMetrics` function.
Every variable name and every operation matches the original JavaScript.

In [13]:
def getMetrics(wyr):
    """
    Python port of Goldman getMetrics(wyr).
    Returns the SDD image for the given water year, masked by water and low_value.
    """

    # Goldman: var start = (wyr - 1) + wyStartDay
    #          var end   = wyr + wyStartDay
    start = str(wyr - 1) + wyStartDay
    end   = str(wyr)     + wyEndDay

    # Goldman: var start_adv = end
    #          var end_adv   = ee.Date(end).advance(30, 'day')
    start_adv = end
    end_adv   = ee.Date(end).advance(30, 'day')

    # Goldman: var adv_days = ee.ImageCollection("MODIS/006/MOD10A1")
    #            .filterDate(start_adv, end_adv).size().getInfo()
    # We use Collection 6.1 (061) — current version of the same product.
    adv_days = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start_adv, end_adv)
        .filterBounds(region_geom)
        .size()
        .getInfo()
    )
    print(f'  adv_days = {adv_days}')

    # Goldman: var MODIS_cloud_list = ee.ImageCollection("MODIS/006/MOD10A1")
    #            .filterDate(start, end_adv)
    #            .map(img => img.expression("(BAND==250)",
    #                         {BAND: img.select('NDSI_Snow_Cover_Class')}))
    #            .toList(400)
    # We add MYD10A1 (Aqua) merged with MOD10A1 (Terra) to reduce cloud gaps.
    # Cloud composite: max across sensors (if either sees cloud → cloud).
    MODIS_cloud_list = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start, end_adv)
        .filterBounds(region_geom)
        .merge(
            ee.ImageCollection('MODIS/061/MYD10A1')
            .filterDate(start, end_adv)
            .filterBounds(region_geom)
        )
        .sort('system:time_start')
        .map(lambda img: img.expression(
            '(BAND == 250)', {'BAND': img.select('NDSI_Snow_Cover_Class')}
        ))
        .toList(400)
    )

    # Goldman: var MODIS_snow_list = ee.ImageCollection("MODIS/006/MOD10A1")
    #            .filterDate(start, end_adv)
    #            .map(img => img.expression(
    #              '(BAND>='+NDSI_threshold+'&&BAND<=100)',
    #              {BAND: img.select('NDSI_Snow_Cover')}))
    #            .toList(400)
    MODIS_snow_list = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start, end_adv)
        .filterBounds(region_geom)
        .merge(
            ee.ImageCollection('MODIS/061/MYD10A1')
            .filterDate(start, end_adv)
            .filterBounds(region_geom)
        )
        .sort('system:time_start')
        .map(lambda img: img.expression(
            '(BAND >= threshold) && (BAND <= 100)',
            {'BAND': img.select('NDSI_Snow_Cover'), 'threshold': NDSI_threshold}
        ))
        .toList(400)
    )

    # Goldman: var water = ee.ImageCollection("GLCF/GLS_WATER")
    #            .map(img => img.expression("(BAND==2)",{BAND:img.select('water')}))
    #            .or().not()
    water = (
        ee.ImageCollection('GLCF/GLS_WATER')
        .map(lambda img: img.expression(
            '(BAND == 2)', {'BAND': img.select('water')}
        ))
        .Or()
        .Not()
    )

    # Goldman: var ndays = MODIS_snow_list.length().getInfo() - adv_days
    ndays = MODIS_snow_list.length().getInfo() - adv_days
    print(f'  ndays = {ndays}')

    # Goldman: var sddCorrection = (ndays < 365+(wyr%4?0:1)) ? 365+(wyr%4?0:1)-ndays : 0
    expected      = 365 + (0 if wyr % 4 else 1)
    sddCorrection = expected - ndays if ndays < expected else 0
    print(f'  sddCorrection = {sddCorrection}')

    # Goldman: initialise stateful images
    sddImage               = ee.Image(0)
    future_snow            = ee.Image(0)
    Snow                   = ee.Image(0)
    accSnow                = ee.Image(0)
    ephemeral_counter      = ee.Image(0)
    noSnowCounter          = ee.Image(0)
    minNoSnow              = ee.Image(minNoSnowdays)
    min_sdays_prior_to_SDD = ee.Image(prior_days)
    sddDetected            = ee.Image(0)

    # Goldman: for(var current_day=ndays+adv_days-1; current_day>=0; current_day--)
    for current_day in range(ndays + adv_days - 1, -1, -1):

        # Goldman: var Cloud = ee.Image(MODIS_cloud_list.get(current_day)).unmask()
        Cloud = ee.Image(MODIS_cloud_list.get(current_day)).unmask()

        # Goldman: future_snow = Cloud.and(future_snow.or(Snow))
        future_snow = Cloud.And(future_snow.Or(Snow))

        # Goldman: Snow = ee.Image(MODIS_snow_list.get(current_day)).unmask()
        Snow = ee.Image(MODIS_snow_list.get(current_day)).unmask()

        # Goldman: if (current_day < ndays)
        if current_day < ndays:

            # Goldman: var snow_occurrence = future_snow.or(Snow)
            snow_occurrence = future_snow.Or(Snow)

            # Goldman: accSnow = accSnow.add(snow_occurrence)
            accSnow = accSnow.add(snow_occurrence)

            # Goldman: min_sdays_prior_to_SDD = min_sdays_prior_to_SDD.where(
            #            snow_occurrence.eq(0).and(
            #              ephemeral_counter.gt(min_sdays_prior_to_SDD)),
            #            ephemeral_counter)
            min_sdays_prior_to_SDD = min_sdays_prior_to_SDD.where(
                snow_occurrence.eq(0).And(
                    ephemeral_counter.gt(min_sdays_prior_to_SDD)
                ),
                ephemeral_counter
            )

            # Goldman: ephemeral_counter = (ephemeral_counter.add(snow_occurrence))
            #                               .multiply(snow_occurrence)
            ephemeral_counter = (
                ephemeral_counter.add(snow_occurrence).multiply(snow_occurrence)
            )

            # Goldman: noSnowCounter = (noSnowCounter.add(snow_occurrence.eq(0)))
            #                           .multiply(sddDetected.eq(0))
            noSnowCounter = (
                noSnowCounter
                .add(snow_occurrence.eq(0))
                .multiply(sddDetected.eq(0))
            )

            # Goldman: sddDetected = (ephemeral_counter.gt(min_sdays_prior_to_SDD))
            #                         .and(noSnowCounter.gt(minNoSnow))
            sddDetected = (
                ephemeral_counter.gt(min_sdays_prior_to_SDD)
                .And(noSnowCounter.gt(minNoSnow))
            )

            # Goldman: minNoSnow = minNoSnow.where(sddDetected, noSnowCounter)
            minNoSnow = minNoSnow.where(sddDetected, noSnowCounter)

            # Goldman: var sdd1 = current_day + sddCorrection
            #          var sdd  = min_sdays_prior_to_SDD.add(ee.Image(sdd1))
            #          sddImage = sddImage.where(sddDetected, sdd)
            sdd1     = current_day + sddCorrection
            sdd      = min_sdays_prior_to_SDD.add(ee.Image(sdd1))
            sddImage = sddImage.where(sddDetected, sdd)

    # Goldman: sddImage = sddImage.where(accSnow.eq(ndays-1), ndays+sddCorrection)
    sddImage = sddImage.where(accSnow.eq(ndays - 1), ndays + sddCorrection)

    # Goldman: scfImage = accSnow.divide(ndays).updateMask(accSnow)
    #                      .rename('ccSCF').updateMask(water)
    scfImage = (
        accSnow.divide(ndays)
        .updateMask(accSnow)
        .rename('ccSCF')
        .updateMask(water)
    )

    # Goldman: sddImage = sddImage.updateMask(sddImage).rename('SDD').updateMask(water)
    sddImage = sddImage.updateMask(sddImage).rename('SDD').updateMask(water)

    # Goldman: var low_end    = ee.Image(low_value)
    #          var lower_mask = scfimg.gte(low_end)
    #          var SDD        = sddimg.updateMask(lower_mask)
    low_end    = ee.Image(low_value)
    lower_mask = scfImage.gte(low_end)
    SDD        = sddImage.updateMask(lower_mask)

    return SDD


print('getMetrics() defined.')

getMetrics() defined.


## Cell 5 — Run and Export

Goldman loops over fire perimeters and exports one clipped image per fire.
We run `getMetrics()` for each year and export one regional raster to Drive.

In [ ]:
export_tasks = []

for wyr in YEARS:
    print(f'\n{"="*50}')
    print(f'Running getMetrics({wyr})')
    print(f'{"="*50}')

    # Goldman: var snowReturn = getMetrics(wyr)
    snowReturn = getMetrics(wyr)

    # Goldman clips per fire and exports. We export the full region.
    # Goldman: Export.image.toDrive({image: snowClip, description: id+'_SDD', ...})
    description = f'SDD_{wyr}_v2'
    task = ee.batch.Export.image.toDrive(
        image          = snowReturn.clip(region_geom),
        description    = description,
        folder         = imagesFolder,
        fileNamePrefix = description,
        region         = region_geom,
        scale          = 500,
        crs            = projection,
        maxPixels      = 1e12,
        fileFormat     = 'GeoTIFF',
    )
    task.start()
    export_tasks.append((wyr, task))
    print(f'  ✓ Export submitted: {description}  (task {task.id})')

print('\nAll tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/tasks')


Running getMetrics(2024)
  adv_days = 30
  ndays = 370
  sddCorrection = 0
  ✓ Export submitted: SDD_2024_v2  (task CX5NYKRGS5GAMWVGHJPCZZDQ)

Running getMetrics(2025)
  adv_days = 30
  ndays = 370
  sddCorrection = 0


## Cell 6 — Monitor Tasks (optional)

In [6]:
import time

def monitor_tasks(tasks, poll_interval_s=60):
    pending = list(tasks)
    while pending:
        still_running = []
        for wyr, task in pending:
            state = task.status()['state']
            if state == 'COMPLETED':
                print(f'  [{wyr}] ✓ COMPLETED')
            elif state == 'FAILED':
                print(f'  [{wyr}] ✗ FAILED — {task.status().get("error_message", "")}')
            elif state == 'CANCELLED':
                print(f'  [{wyr}] — CANCELLED')
            else:
                print(f'  [{wyr}] {state}')
                still_running.append((wyr, task))
        pending = still_running
        if pending:
            time.sleep(poll_interval_s)
    print('Done.')

monitor_tasks(export_tasks)

  [2024] RUNNING
  [2025] READY
  [2024] RUNNING
  [2025] READY
  [2024] RUNNING
  [2025] READY
  [2024] RUNNING
  [2025] READY
  [2024] ✓ COMPLETED
  [2025] RUNNING
  [2025] RUNNING
  [2025] RUNNING
  [2025] RUNNING
  [2025] RUNNING
  [2025] ✓ COMPLETED
Done.
